# Part 8 · Notebook 01 — The research log and the event-driven engine

**Sessions:** S1 (Backtesting philosophy & taxonomy) · S2 (Event-driven architecture) · [Lesson plan](../../docs/lessons/PART_08_BACKTESTING_RISK_PORTFOLIO.md) · graded labs in [`labs/part08/`](../../labs/part08/)

**You will:**
1. Give every backtest configuration a stable fingerprint, and log every run.
2. Write the ledger that turns fills into cash, positions and equity (with futures multipliers).
3. Order same-timestamp events deterministically.
4. Run the event-driven engine on a strategy from Part 7.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic with a known truth (known regimes, known Sharpe ratios, pure noise), so every statistic can be checked against reality and every discovery against luck.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p8lib.py is in notebooks/part08/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p8lib as p

p.use_course_style()
import hashlib, json

## 1. Every run leaves a trace

A backtest is an experiment, and the most dangerous number in research is the one you don't record: how many things you tried. The Deflated Sharpe Ratio (notebook 07) needs **all** the trials, including the failures. So every run is logged with a fingerprint of its configuration: the first 12 hex characters of the SHA-256 of `json.dumps(config, sort_keys=True)`. Sorting the keys makes the hash independent of the order you wrote them in.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def config_hash(config):
    return ...                                    # ✍️ sha256 of the sorted JSON, first 12 hex characters

configs = [{"strategy": "sma_cross", "params": {"fast": 20, "slow": 100}, "data_version": "v1"},
           {"data_version": "v1", "params": {"slow": 100, "fast": 20}, "strategy": "sma_cross"},     # same, reordered
           {"strategy": "sma_cross", "params": {"fast": 20, "slow": 101}, "data_version": "v1"}]
mine = [p.attempt(config_hash, c) for c in configs]
mine = p.check("config_hash", mine, [p.config_hash(c) for c in configs])
mine

In [ ]:
bars = p.regime_market()
o, c = bars.open.to_numpy(), bars.close.to_numpy()
log = p.ResearchLog()                                      # sqlite in memory; a file in real research
for fast in (5, 10, 20, 40):
    for slow in (50, 100, 150, 200, 250):
        pnl = p.first_look_pnl(p.sma_cross_signal(c, fast, slow), o)
        log.record("sma_cross", {"fast": fast, "slow": slow}, "regime_market:v1", p.sharpe(pnl), p.max_drawdown(pnl)[0], len(pnl))
trials = log.trials("sma_cross")
print(f"{len(trials)} trials logged; best Sharpe {trials.sharpe.max():.2f}, median {trials.sharpe.median():.2f}")
trials.sort_values("sharpe", ascending=False).head()

The best of 20 tries is what gets reported; the other 19 are what make it believable, or not.

## 2. The ledger

The engine's portfolio keeps cash, positions, last prices and **contract multipliers** (ES futures: 50 dollars per point). A fill of `qty` at `price` costs `qty × price × multiplier + fee` in cash (a sale, with negative `qty`, brings cash in); fees accumulate in `self.fees`. Equity is cash plus every position marked at its last price, times its multiplier.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
class MyPortfolio(p.Portfolio):
    def on_fill(self, sym, qty, price, fee=0.0):
        m = self.mult.get(sym, 1.0)
        ...                                       # ✍️ update cash, fees and the position

    @property
    def equity(self):
        return ...                                # ✍️ cash + Σ qty × last price × multiplier

def replay(cls):
    pf = cls(1_000_000, multipliers={"ES": 50})
    out = []
    for kind, *args in [("fill", "SPY", 100, 500.0, 1.0), ("fill", "ES", 2, 5000.0, 4.5), ("mark", "SPY", 505.0),
                        ("mark", "ES", 5010.0), ("fill", "SPY", -50, 506.0, 1.0), ("mark", "ES", 4990.0)]:
        pf.on_fill(*args) if kind == "fill" else pf.mark(*args)
        if kind == "fill":
            pf.mark(args[0], args[2])
        out.append((round(pf.cash, 2), round(pf.equity, 2) if pf.equity is not Ellipsis else Ellipsis))
    return out

mine = p.attempt(replay, MyPortfolio)
mine = p.check("ledger", mine, replay(p.Portfolio))
pd.DataFrame(mine, columns=["cash", "equity"])

Two ES contracts cost $500,000 of *notional* here only because this toy ledger books futures like stock. A real futures ledger books margin and daily variation instead; the equity arithmetic (P&L = Δprice × qty × 50) is the same.

## 3. Deterministic event order

The engine is a priority queue of events. Events with the **same timestamp** must be processed in a fixed order, or two runs of the same backtest can differ: fills first (priority 0), then bars (1), then timers (2), and within a tie, the order they were added (`seq`). Write the sort key.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
PRIORITY = {"fill": 0, "bar": 1, "timer": 2}
events = [{"ts": 2, "kind": "timer", "seq": 0}, {"ts": 1, "kind": "bar", "seq": 1}, {"ts": 2, "kind": "bar", "seq": 2},
          {"ts": 2, "kind": "fill", "seq": 3}, {"ts": 1, "kind": "timer", "seq": 4}, {"ts": 2, "kind": "fill", "seq": 5}]

def event_key(e):
    return ...                                    # ✍️ a tuple: timestamp, then priority, then sequence number

mine = p.attempt(lambda: [(e["ts"], e["kind"], e["seq"]) for e in sorted(events, key=event_key)])
q = p.EventQueue()
for e in events:
    q.push(e["ts"], PRIORITY[e["kind"]], e["kind"], e["seq"])
ref = []
while q:
    ev = q.pop()
    ref.append((ev.ts, ev.kind, ev.data))
mine = p.check("event order", mine, ref)
mine

## 4. The engine

`p.run_backtest` puts it together for one symbol: at each close the strategy's target weight becomes a share order; the order fills at the **next open** (with slippage and commission if given); positions are marked at the open and the close. Same strategy, same bars as Part 7.

In [ ]:
sig = p.sma_cross_signal(c, 20, 100)
res = p.run_backtest(bars, sig, slippage_bps=2.0, commission=p.ib_fixed_commission)
fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
axes[0].plot(res.index, res.equity / 1e6); axes[0].set_ylabel("equity, $m")
axes[1].plot(res.index, res.position); axes[1].set_ylabel("shares")
axes[0].set_title("SMA 20/100 through the event-driven engine"); plt.show()
print(f"final equity ${res.equity.iloc[-1]:,.0f}; commissions paid ${res.fees.iloc[-1]:,.0f}")

## Wrap-up

* Log every run, with a configuration hash and a data version.
* The ledger handles multipliers and fees; equity is cash plus marked positions.
* Same-timestamp events have a fixed order, so every run replays identically.
* Graded version: `labs/part08/week25_engine` (DuckDB research log, event queue, ledger, fills, the engine).